# WDL — Module 1 · Session 4 — Reading a river gauge record
### Practice activity

| | |
|---|---|
| **Course** | Water Data Literacy: Rainfall, Floods and Climate Change |
| **Edition** | Udemy |
| **Instructor** | Dr Arpan Pradhan |
| **Module** | Module 1 — The Water Cycle as Data |
| **Session** | Session 4 of 30 |
| **Topics** | River stage; warning and danger levels; rainfall–river response; lag time; rise and recession |
| **Water hook** | *When a river crosses the danger mark* |

---
> **How to use this notebook.** No coding needed. Read each explanation, then press ▶ on the box below it
> (or click the box and press `Shift + Enter`). Some boxes have sliders, number boxes or dropdowns: change them, then press ▶ again.
> Go to **File → Save a copy in Drive** first, so the notebook is yours.

Measure the September flood at the sample station. Use the table to find each answer, then check it.

In [ ]:
#@title ▶ Step 0 · Get ready (press ▶ once before anything else)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from IPython.display import display, Markdown
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 110, "font.size": 11,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, RED, GREY, TEAL = "#2C6E9B", "#DD8452", "#C44E52", "#8C8C8C", "#1F7A8C"
def ok(msg="Correct!"):  display(Markdown(f"✅ **{msg}**"))
def no(msg="Not quite. Check your working and try again."): display(Markdown(f"❌ {msg}"))
def close(a, b, tol=0.02):  # within 2% (or 0.01 for very small numbers)
    try: return abs(float(a) - float(b)) <= max(abs(float(b)) * tol, 0.01)
    except (TypeError, ValueError): return False

DATA_URL = "https://raw.githubusercontent.com/PradhanArpan/waterliteracy/main/data/M1/sample_station_2023.csv"
def _make_sample_station():
    rng = np.random.default_rng(2023)
    days = pd.date_range("2023-01-01", "2023-12-31", freq="D")
    p_wet  = [0.03, 0.03, 0.08, 0.20, 0.35, 0.40, 0.45, 0.48, 0.55, 0.50, 0.30, 0.08]
    mean_r = [3, 3, 6, 9, 10, 7, 7, 8, 11, 11, 9, 4]
    rain = []
    for d in days:
        m = d.month - 1
        rain.append(rng.gamma(0.8, mean_r[m] / 0.8) if rng.random() < p_wet[m] else 0.0)
    rain = np.round(np.array(rain), 1)
    rain[days.get_loc(pd.Timestamp("2023-09-21"))] = 96.4
    rain[days.get_loc(pd.Timestamp("2023-09-22"))] = 58.2
    level, soil, channel = [], 0.0, 0.0
    for r in rain:
        soil = soil * 0.6 + max(r - 5, 0) * 0.5
        channel = channel * 0.82 + soil * 0.022
        level.append(round(1.8 + channel, 2))
    return pd.DataFrame({"date": days, "rain_mm": rain, "level_m": level})
try:
    station = pd.read_csv(DATA_URL, parse_dates=["date"])
except Exception:
    station = _make_sample_station()   # works offline too: same numbers
station = station.set_index("date")
rainfall, river = station[["rain_mm"]], station[["level_m"]]
WARNING_LEVEL, DANGER_LEVEL = 3.6, 4.2
display(Markdown("✅ **Ready.** Sample station loaded: one year (2023) of daily rainfall and river level. "
                 "*Illustrative, synthetic data for practice; not an official IMD or CWC record.*"))

In [ ]:
#@title ▶ Flood table: 15 Sep to 10 Oct
t = station.loc["2023-09-15":"2023-10-10"].copy()
t["above warning?"] = np.where(t.level_m > WARNING_LEVEL, "yes", "")
t["above danger?"] = np.where(t.level_m > DANGER_LEVEL, "yes", "")
t.index = t.index.strftime("%d %b")
display(t)

In [ ]:
#@title ▶ Check Q1 · Highest river level of the year (m)
peak_m = 0 #@param {type:"number"}
ans = river.level_m.max()
ok(f"Correct: {ans:.2f} m") if close(peak_m, ans, tol=0.005) else no("Hint: find the largest number in the level_m column.")

In [ ]:
#@title ▶ Check Q2 · Lag time (days between the heaviest rain and the river peak)
lag_days = 0 #@param {type:"integer"}
ans = (river.level_m.idxmax() - rainfall.rain_mm.idxmax()).days
ok(f"Correct: {ans} days") if lag_days == ans else no("Hint: find the date of the most rain and the date of the highest level, then count the days between.")

In [ ]:
#@title ▶ Check Q3 · Days above the danger level (4.2 m)
danger_days = 0 #@param {type:"integer"}
ans = int((river.level_m > DANGER_LEVEL).sum())
ok(f"Correct: {ans} days") if danger_days == ans else no("Hint: count the 'yes' entries in the 'above danger?' column.")

In [ ]:
#@title ▶ Check Q4 · Days from the peak until the river fell back below the warning level
days_to_safety = 0 #@param {type:"integer"}
lv = river.level_m; pk = lv.idxmax()
after = lv[pk:]; below = after[after <= WARNING_LEVEL].index[0]
ans = (below - pk).days
ok(f"Correct: {ans} days (back below warning on {below:%d %b})") if days_to_safety == ans else no("Hint: start at the peak date and count forward to the first day with no 'yes' under 'above warning?'.")

In [ ]:
#@title ▶ Check Q5 · Did the river rise faster than it fell?
answer = "choose" #@param ["choose", "Yes, it rose faster", "No, it fell faster", "Both took the same time"]
lv = river.level_m; pk = lv.idxmax()
start = lv[:pk][lv[:pk] <= WARNING_LEVEL].index[-1]
after = lv[pk:]; end = after[after <= WARNING_LEVEL].index[0]
rise, fall = (pk - start).days, (end - pk).days
msg = f"From the warning level it took {rise} days to rise to the peak and {fall} days to fall back."
ok("Correct. " + msg) if answer == "Yes, it rose faster" else no("Look again at the table. " if answer == "choose" else "Not quite. " + msg)